In [3]:
import pandas as pd

# 1. CSV 파일 불러오기 (파일명을 'input_data.csv'라고 가정했습니다. 실제 파일명으로 수정하세요.)
# 만약 파일의 구분자가 탭(\t)이라면 sep='\t' 옵션을 추가해야 합니다.
try:
    df = pd.read_csv('Sample01_AIS_new_category_final.csv')

    # 2. timestamp 컬럼을 datetime 객체로 변환 (정확한 정렬을 위함)
    df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', utc=True)

    # 3. timestamp 기준 내림차순 정렬 (ascending=False)
    df_sorted = df.sort_values(by='timestamp', ascending=True)

    # 4. 정렬된 데이터를 AIS_category_file.csv로 저장 (인덱스 제외)
    df_sorted.to_csv('Sample01_AIS_new_category_final_asc.csv', index=False, encoding='utf-8-sig')

    print("정렬 및 저장이 완료되었습니다: Sample01_AIS_new_category_final_asc.csv")

except FileNotFoundError:
    print("파일을 찾을 수 없습니다. 파일명을 확인해주세요.")
except Exception as e:
    print(f"오류가 발생했습니다: {e}")

정렬 및 저장이 완료되었습니다: Sample01_AIS_new_category_final_asc.csv


In [16]:
import streamlit as st
import pandas as pd
import pydeck as pdk
import time
import os
import json

In [15]:
file_path = 'AIS_category_file.csv'
df = pd.read_csv(file_path)
df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', utc=True)

speed_factor=10.0
print(f"--- 시뮬레이션 시작 (배속: {speed_factor}x) ---")

prev_time = None

for index, row in df.iterrows():
    current_data_time = row['timestamp']

    if prev_time is not None:
        # 2. 실제 데이터 간의 시간 차이 계산 (초 단위)
        time_diff = (current_data_time - prev_time).total_seconds()
        
        # 3. 배속에 따른 대기 시간 계산
        # 만약 데이터 간격이 10초이고 10배속이면 1초만 대기
        sleep_time = time_diff / speed_factor
        
        if sleep_time > 0:
            time.sleep(sleep_time)

    # 4. 패킷 출력
    packet = row.to_dict()
    # 시간 형식 가독성을 위해 문자열로 변환하여 출력
    packet['timestamp'] = packet['timestamp'].strftime('%Y-%m-%d %H:%M:%S')
    print(f"[전송 시간: {packet['timestamp']}] {packet['ShipName']} ({packet['mmsi']})")

    prev_time = current_data_time

print("--- 시뮬레이션 종료 ---")

--- 시뮬레이션 시작 (배속: 10.0x) ---
[전송 시간: 2022-12-01 00:00:01] LST-683 향로봉 (563161700)
[전송 시간: 2022-12-01 00:00:02] LST-683 향로봉 (563161700)
[전송 시간: 2022-12-01 00:00:03] FF-959 부산 (305062000)
[전송 시간: 2022-12-01 00:00:03] DDH-971 광개토대왕 (305089000)
[전송 시간: 2022-12-01 00:00:03] LST-675 개봉 (371473000)
[전송 시간: 2022-12-01 00:00:04] FFG-818 대구 (352002106)
[전송 시간: 2022-12-01 00:00:05] PKG-726 한문식 (636020104)
[전송 시간: 2022-12-01 00:00:05] AGS- 신세기 (413379570)
[전송 시간: 2022-12-01 00:00:06] PKG-722 임병래 (273331810)
[전송 시간: 2022-12-01 00:00:07] AST-688 일천봉 (256011000)
[전송 시간: 2022-12-01 00:00:09] DDH-976 문무대왕 (477016800)
[전송 시간: 2022-12-01 00:00:09] PKG-713 조천형 (477655100)
[전송 시간: 2022-12-01 00:00:10] FFG-816 충북 (273347430)
[전송 시간: 2022-12-01 00:00:11] DDH-973 양만춘 (538008114)
[전송 시간: 2022-12-01 00:00:14] SS-072 손원일 (413956000)
[전송 시간: 2022-12-01 00:00:14] AGS- 신천지 (352001086)
[전송 시간: 2022-12-01 00:00:21] MSH- 옹진 (440467000)
[전송 시간: 2022-12-01 00:00:21] FFG-819 경남 (413482000)
[전송 시간: 2022-12-01 00:00:21] LS

KeyboardInterrupt: 

In [17]:
file_path = 'AIS_Korea_Dec_08.csv'
df = pd.read_csv(file_path)
count = df['ship_and_cargo_type'].nunique()
print(count) 

102


In [19]:
import pandas as pd
import numpy as np

# 1. 파일 읽기
file_path = 'AIS_Korea_Dec_08.csv'
df = pd.read_csv(file_path)

# 2. 전처리: 숫자가 아닌 값(빈칸, 문자 등)을 처리하기 위해 숫자형으로 변환
# 변환할 수 없는 값은 NaN이 됩니다.
s_numeric = pd.to_numeric(df['ship_and_cargo_type'], errors='coerce')

# 3. 조건 설정: 십의 자리가 1~8인 경우 (즉, 숫자가 10 이상 90 미만인 경우)
# 이 범위를 벗어나는 0~9, 90~99, 100 이상의 수, NaN은 모두 False가 됩니다.
condition = (s_numeric >= 10) & (s_numeric < 90)

# 4. 필드 생성 (오류 방지 로직)
# np.where의 결과가 float 형태일 수 있으므로, fillna(9)를 먼저 수행합니다.
df['unique_type'] = np.where(condition, s_numeric // 10, 9)

# 5. 마지막에 정수형으로 변환 (NaN이 이미 9로 채워졌으므로 안전함)
df['unique_type'] = df['unique_type'].astype(int)

# 5. 결과 저장 (새로운 파일명으로 저장하거나 기존 파일에 덮어쓰기)
output_path = 'AIS_Korea_Dec_08_updated.csv'
df.to_csv(output_path, index=False)

# 결과 확인용 출력
print(f"새로운 필드가 추가된 파일이 '{output_path}'로 저장되었습니다.")
print(df['unique_type'].value_counts().sort_index())

새로운 필드가 추가된 파일이 'AIS_Korea_Dec_08_updated.csv'로 저장되었습니다.
unique_type
1      4037
2       815
3    141087
4      1312
5    127718
6     29791
7    398893
8    155901
9    320959
Name: count, dtype: int64


In [20]:
import pandas as pd

# 1. CSV 파일 불러오기 (파일명을 'input_data.csv'라고 가정했습니다. 실제 파일명으로 수정하세요.)
# 만약 파일의 구분자가 탭(\t)이라면 sep='\t' 옵션을 추가해야 합니다.
try:
    df = pd.read_csv('AIS_Korea_Dec_08_updated.csv')

    # 2. timestamp 컬럼을 datetime 객체로 변환 (정확한 정렬을 위함)
    df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', utc=True)

    # 3. timestamp 기준 내림차순 정렬 (ascending=False)
    df_sorted = df.sort_values(by='timestamp', ascending=True)

    # 4. 정렬된 데이터를 AIS_category_file.csv로 저장 (인덱스 제외)
    df_sorted.to_csv('AIS_Korea_Dec_08_updated_time.csv', index=False, encoding='utf-8-sig')

    print("정렬 및 저장이 완료되었습니다: AIS_Korea_Dec_08_updated_time.csv")

except FileNotFoundError:
    print("파일을 찾을 수 없습니다. 파일명을 확인해주세요.")
except Exception as e:
    print(f"오류가 발생했습니다: {e}")

정렬 및 저장이 완료되었습니다: AIS_Korea_Dec_08_updated_time.csv
